# Processing MESACLIP data - sea ice monthly, single ensembles

* **Description**: Reads in MESACLIP historical and crops to just the Arctic
* **Creator**: Alice DuVivier
* **Date**: December 2025

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [2]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='2:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:274: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

In [3]:
client
# cluster.close()

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.179:43653,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/duvivier/PBS/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Set MESACLIP ensemble information

In [4]:
# Historical
path_base = '/gdex/data/d651007/'
case_base = 'b.e13.BHISTC5.ne120_t12.cesm-ihesp-'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['sehires38-1850-2005.001', 'hires1.0.30-1920-2005.002', 
            'hires1.0.30-1920-2005.003', 'hires1.0.44-1920-2005.004', 
            'hires1.0.44-1920-2005.005', 'hires1.0.45-1920-2005.006', 
            'hires1.0.46-1920-2005.007', 'hires1.0.46-1920-2005.008',
            'hires1.0.46-1920-2005.009', 'hires1.0.46-1920-2005.010']

nens = len(ens_mems)

In [5]:
# set the groups of years since we have to load them separately (UGH)
# 1920s, 1930s, 1940s, 1950s, 1960s, 1970s, 1980s, 1990s, 2000s
year_sets = ['192','193','194','195','196',
             '197','198','199','200']

nyears = len(year_sets)

## Load in SEA ICE Data

In [6]:
# Choose variable to load
# 'aice', 'hi', 
#'aicen001','aicen002','aicen003','aicen004','aicen005'
#'vicen001','vicen002','vicen003','vicen004','vicen005'

var_in = 'aicen001'

In [7]:
# set output directory
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/monthly/'+var_in+'/'

In [9]:
# loop through each ensemble and each set of years

# loop through ensembles
for n in np.arange(0,nens,1)[0:2]:
    # loop through year groupings
    for y in np.arange(0,nyears,1):
        ens_in = ens_mems[n]
        yr_pre = year_sets[y]
        print('loading ensemble: '+ens_in+' for years '+yr_pre+'0s')
        # set case
        case = case_base + ens_in
        # set data directory
        path = path_base+case+'/ice/proc/tseries/month_1'
        # get list of files
        files = sorted(glob(f'{path}/{case}.cice.h.{var_in}.{yr_pre}???-??????.nc'))       
        # load files
        ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                           concat_dim="time", combine='nested', decode_times=True)
        # take an average over the time bounds to get the right time dimension
        ds_tmp["time"] = ds_tmp.time_bounds.compute().mean(dim="d2")

        # subset to Arctic Only
        ds_tmp = ds_tmp.isel(nj=slice(1750,2400))

        # save and export data
        # set output file name
        nn = n+1
        if nn < 10:
            ensnum = '00'+str(nn)
        elif nn >= 10:
            ensnum = '010'

        file_out = 'MESACLIP_HIST_'+ensnum+'_'+yr_pre+'0-'+yr_pre+'9'+'_'+var_in+'_arctic_monthly.nc'
        fout = path_out + file_out
        #print(fout)

        # make output array
        ds_out = ds_tmp
        # change the attributes
        ds_out.attrs['author'] = 'Alice DuVivier'
        ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
        ds_out.attrs['contents'] = f'Subset of historical MESACLIP Data'
        # calculate the size of the dataset in GB
        size_gb = ds_out.nbytes / (1024**3)
        print(f"The dataset is approximately {size_gb:.2f} GB.")

        # Export the dataset to NetCDF with all attributes and coordinates
        print('Writing post processed file')
        ds_out.to_netcdf(fout)

print("Done!")

loading ensemble: sehires38-1850-2005.001 for years 1920s
The dataset is approximately 1.33 GB.
Writing post processed file
loading ensemble: sehires38-1850-2005.001 for years 1930s
The dataset is approximately 1.33 GB.
Writing post processed file


Task exception was never retrieved
future: <Task finished name='Task-527269' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-527303' coro=<Client._gather.<locals>.wait() done, defined at /glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/distributed/client.py:2208> exception=AllExit()>
Traceback (most recent call last):
  File "/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/distributed/client.py", line 2217, in wait
    raise AllExit()
distributed.client.AllExit
Task exception was never retrieved
future: <Task finished name='Task-527306' coro=<Client._gather.<loc

In [ ]:
print('closing cluster')
cluster.close()